# 04 — Pandas for signal data

Pandas is the odd one out in this stack. Signals and Systems is mathematics on arrays, and
NumPy already does that. So why learn pandas?

Because **the maths is never the hard part of a real measurement.** The hard part is that
your instrument logged comment banners in the middle of the file, used `-999.0` to mean "no
reading", dropped 2 seconds of samples, and wrote timestamps in a format nobody agreed on.
Pandas is the tool for the two hours before you get to call `scipy.signal.butter`.

Its second job is **bookkeeping over experiments**: forty filter designs, six trials each,
four metrics per trial. That is a table, not an array, and it is exactly what a DataFrame is
for.

**Rule of thumb:** pandas for *loading, cleaning, aligning, and comparing*; NumPy/SciPy for
*computing*. Pull `.to_numpy()` out of the DataFrame the moment you start doing real DSP,
and put results back into a DataFrame when you want to compare them.

---

## Contents

| § | Topic |
|---|-------|
| 1 | Series and DataFrame: just enough structure |
| 2 | Loading a clean CSV |
| 3 | Loading a *messy* instrument log |
| 4 | Cleaning: sentinels, types, and validity flags |
| 5 | Missing data and dropouts |
| 6 | The DatetimeIndex and `resample` |
| 7 | Rolling windows |
| 8 | Multi-channel data: `accelerometer.csv` |
| 9 | Bridging to SciPy |
| 10 | GroupBy: comparing many experiments |
| 11 | Reshaping: `pivot`, `melt`, wide vs. long |
| 12 | Building a results table from scratch |
| 13 | Exporting |
| 14 | Performance notes |
| 15 | Exercises |

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({
    "figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

print("pandas", pd.__version__)
DATA = "../data"

---
## 1. Series and DataFrame: just enough structure

- **Series** — a 1-D array *with an index*. A single channel.
- **DataFrame** — a dict of Series sharing one index. A multi-channel recording, or a table
  of experiment results.

The index is the whole point. It is what lets pandas align two recordings that were sampled
at different times without you writing any interpolation code.

In [ ]:
fs = 100.0
t = np.arange(500) / fs
x = np.sin(2 * np.pi * 3 * t)

s = pd.Series(x, index=pd.Index(t, name="time_s"), name="channel_a")
print(s.head())
print()
print("dtype :", s.dtype)
print("shape :", s.shape)
print("index :", s.index[:3].to_numpy(), "...")

In [ ]:
df = pd.DataFrame({
    "channel_a": np.sin(2 * np.pi * 3 * t),
    "channel_b": np.sin(2 * np.pi * 3 * t + np.pi / 3),
    "channel_c": signal.square(2 * np.pi * 1 * t),
}, index=pd.Index(t, name="time_s"))

print(df.head())
print()
print(df.describe().round(4))

> **`.describe()` is your first move on any new dataset.** Count, mean, std, min, quartiles,
> max — per column, in one call. A `count` lower than the others tells you a channel has
> missing values. A `min` of `-999` tells you someone used a sentinel.

In [ ]:
# Alignment: the feature that justifies the index.
a = pd.Series([1.0, 2.0, 3.0], index=[0.0, 0.1, 0.2])
b = pd.Series([10.0, 20.0, 30.0], index=[0.1, 0.2, 0.3])

print("a + b (aligned on the index, not on position):")
print(a + b)
print("\nNumPy would have refused or silently added the wrong pairs.")

---
## 2. Loading a clean CSV

The easy case first, so the messy case has something to contrast with.

In [ ]:
ecg = pd.read_csv(f"{DATA}/ecg_like.csv")
print(ecg.head())
print()
ecg.info()          # info() prints directly and returns None -- do not wrap it in print()

In [ ]:
# Use the time column as the index -- now slicing by time works directly.
ecg = ecg.set_index("time_s")
fs_ecg = round(1 / np.mean(np.diff(ecg.index.to_numpy())))
print("sampling rate:", fs_ecg, "Hz")

# .loc slices by *label*, and for a numeric index that means by time in seconds.
segment = ecg.loc[2.0:5.0]
print(f"\nsamples between t=2 s and t=5 s: {len(segment)}")

fig, ax = plt.subplots(figsize=(10, 2.8))
segment["mV"].plot(ax=ax)          # pandas plots against the index automatically
ax.set_ylabel("mV"); ax.set_title("ecg_like.csv, 2-5 s")

> **`.loc[2.0:5.0]` is inclusive of both endpoints**, unlike Python slicing and unlike
> `.iloc`. This surprises everyone once. `.loc` = by label, endpoints included.
> `.iloc` = by position, endpoint excluded, like normal Python.

---
## 3. Loading a *messy* instrument log

Now open `data/bench_capture.txt` and look at what a real instrument actually produces.

In [ ]:
with open(f"{DATA}/bench_capture.txt") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 12:
            break

Everything about that file is hostile:

- eight lines of `#` comments before the data
- a **blank line** after them
- `;` as the delimiter, with spaces around it
- no header row — the column names are buried in a comment
- a `# --- keepalive ---` banner reappearing every 1500 rows, *in the middle of the data*
- `NaN` in one place and the sentinel `-999.0` in another, both meaning "no reading"
- a `status` column that tells you which rows to distrust

`read_csv` handles all of it, if you know which arguments to reach for.

> A first attempt that looks right but is not:

In [ ]:
attempt = pd.read_csv(
    f"{DATA}/bench_capture.txt",
    sep=";",
    comment="#",
    header=None,
    names=["index", "time_s", "volts", "temp_c", "status"],
    skipinitialspace=True,
    na_values=["NaN", "-999.0"],
)
print(attempt.dtypes)
print("\nvolts came back as:", attempt['volts'].dtype, "<- that should have been float64")
print("\nWhy? Look at the raw field values:")
print([repr(v) for v in attempt['volts'].head(3)])

The fields are separated by `" ; "` — a semicolon **with a space on each side**.
`skipinitialspace=True` removes the leading space but leaves the *trailing* one, so the
sentinel arrives as `'-999.0 '`, which does not match the `'-999.0'` in `na_values`. One
unmatched string is enough to force the whole column to `object` dtype, and then every
numeric operation downstream fails in a confusing place.

The fix is a **regex separator** that absorbs whitespace on both sides:

In [ ]:
raw = pd.read_csv(
    f"{DATA}/bench_capture.txt",
    sep=r"\s*;\s*",          # regex: a semicolon with any surrounding whitespace
    engine="python",         # regex separators require the Python engine
    comment="#",             # drop anything from '#' onward -- kills the banners too
    skip_blank_lines=True,
    header=None,             # there is no header row
    names=["index", "time_s", "volts", "temp_c", "status"],
    na_values=["NaN", "nan", "-999.0", "-999"],   # both sentinels become NaN
)

print(raw.dtypes)
print()
print(raw.head())
print(f"\n{len(raw)} rows loaded, {raw['volts'].isna().sum()} volts values are NaN")

> **Three things earned their place there.**
>
> **`comment="#"` did double duty.** It skipped the header block *and* removed the keepalive
> banners scattered through the data — the case a naive `skiprows=8` would have failed on.
> Reach for `comment=` rather than counting lines.
>
> **`na_values=` is where you declare your sentinels.** Every lab has its own convention
> (`-999`, `-9999`, `NULL`, `9.9e37`). Declaring them at read time means every downstream
> operation treats them correctly, instead of you finding a `-999` inside your mean six steps
> later.
>
> **Always check `.dtypes` after loading.** An unexpected `object` column is pandas telling
> you that something in that column did not parse. It is the single most useful five-second
> check in this notebook.

In [ ]:
# The status column tells us which rows to trust. Always check what values exist.
print(raw["status"].value_counts())
print()
print("rows with a non-OK status:")
print(raw[raw["status"] != "OK"].head(10))

---
## 4. Cleaning: sentinels, types, and validity flags

There are two philosophies about bad samples: **drop them** or **mark them NaN and keep the
time base**. For signal processing the second is almost always right — dropping rows silently
changes your sampling rate.

In [ ]:
df = raw.copy()

# Strings can carry invisible whitespace. Strip before comparing.
df["status"] = df["status"].str.strip()

# Mark every non-OK sample as missing, keeping the row so the time base stays uniform.
df.loc[df["status"] != "OK", "volts"] = np.nan

print(f"total samples     : {len(df)}")
print(f"valid samples     : {df['volts'].notna().sum()}")
print(f"missing samples   : {df['volts'].isna().sum()}  ({df['volts'].isna().mean():.2%})")

# Confirm the time base is still uniform.
dt = np.diff(df["time_s"].to_numpy())
print(f"\nsample interval: mean {dt.mean():.6f} s, std {dt.std():.2e} s -> uniform")
print(f"inferred fs    : {1 / dt.mean():.1f} Hz")

In [ ]:
df = df.set_index("time_s")

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(df.index, df["volts"], lw=0.6)
bad = df[df["volts"].isna()]
ax[0].plot(bad.index, np.zeros(len(bad)), "rx", ms=5, label=f"{len(bad)} bad samples")
ax[0].set_ylabel("volts"); ax[0].legend()
ax[1].plot(df.index, df["temp_c"], lw=0.8, color="tab:orange")
ax[1].set_ylabel("probe temp [°C]"); ax[1].set_xlabel("time [s]")
ax[0].set_title("bench_capture.txt after cleaning")
fig.tight_layout()

### Downcasting to save memory

For a 4000-row file this is irrelevant. For a 50-million-sample capture it is the difference
between fitting in RAM and not.

In [ ]:
before = df.memory_usage(deep=True).sum()

compact = df.copy()
compact["volts"] = compact["volts"].astype("float32")
compact["temp_c"] = compact["temp_c"].astype("float32")
compact["status"] = compact["status"].astype("category")   # few unique values -> category

after = compact.memory_usage(deep=True).sum()
print(f"before : {before / 1024:8.1f} KB")
print(f"after  : {after / 1024:8.1f} KB   ({100 * (1 - after / before):.0f}% smaller)")
print("\nNote: 'category' is the big win for repeated strings like status flags.")

> **A caution on `float32`.** It has about 7 decimal digits of precision. That is fine for
> raw ADC data (which rarely exceeds 24 bits) but can bite in an accumulating computation —
> a long convolution or a high-order IIR filter. Store compactly, compute in `float64`.

---
## 5. Missing data and dropouts

Once bad samples are `NaN`, you have four options, and the right one depends on why they are
missing and what you plan to do next.

In [ ]:
v = df["volts"]

strategies = {
    "original (NaN kept)":  v,
    "ffill (hold last)":    v.ffill(),
    "linear interpolate":   v.interpolate(method="linear"),
    "cubic interpolate":    v.interpolate(method="cubic"),
    "fill with mean":       v.fillna(v.mean()),
}

# Find a stretch containing several gaps so the difference is visible.
gap_idx = v.isna().to_numpy().nonzero()[0][3]
lo, hi = max(0, gap_idx - 30), gap_idx + 30

fig, ax = plt.subplots(figsize=(10, 3.4))
for name, s in strategies.items():
    style = "o-" if name.startswith("original") else "-"
    ax.plot(s.index[lo:hi], s.iloc[lo:hi], style, ms=4, lw=1.2, label=name, alpha=0.85)
ax.set_xlabel("time [s]"); ax.set_ylabel("volts"); ax.legend(fontsize=8)
ax.set_title("Five ways to handle a dropout")

| strategy | good for | bad because |
|----------|----------|-------------|
| keep `NaN` | statistics (`mean`, `std` skip NaN automatically) | **every SciPy filter returns all-NaN** if fed one |
| `ffill` | slowly-varying signals, sensor state | creates flat steps → broadband spectral artefacts |
| `interpolate("linear")` | short gaps in a smooth signal | the sensible default |
| `interpolate("cubic")` | short gaps, smooth derivative matters | can overshoot wildly across long gaps |
| `fillna(mean)` | almost nothing | injects a step to the mean; distorts the spectrum |

**The rule that matters:** an interpolated sample is fabricated data. Interpolate short gaps
so your filters run, but track how many samples you invented and say so in your report.

In [ ]:
# Why NaN and SciPy do not mix.
b, a = signal.butter(4, 0.1)
with_nan = signal.lfilter(b, a, v.to_numpy())
interp = signal.lfilter(b, a, v.interpolate().to_numpy())

print(f"filtering raw (with NaN)  -> {np.isnan(with_nan).sum()} of {len(with_nan)} outputs are NaN")
print(f"filtering interpolated    -> {np.isnan(interp).sum()} NaN")
print("\nOne NaN poisons everything downstream of it in an IIR filter.")

In [ ]:
# Limit how far interpolation is allowed to reach. Long gaps stay NaN, honestly.
gappy = v.copy()
gappy.iloc[500:560] = np.nan          # a 60-sample dropout

short_only = gappy.interpolate(limit=10)
print(f"gap of 60 samples, limit=10 -> {short_only.iloc[500:560].isna().sum()} still NaN")
print("This is the honest behaviour: fill what you can defend, leave the rest.")

---
## 6. The DatetimeIndex and `resample`

When your data has real timestamps, a `DatetimeIndex` unlocks `resample` — a groupby over
time bins that handles irregular sampling, gaps, and unit conversion in one call.

In [ ]:
acc = pd.read_csv(f"{DATA}/accelerometer.csv", parse_dates=["timestamp"])
acc = acc.set_index("timestamp")

print(acc.head())
print()
print("index type:", type(acc.index).__name__)
print("duration  :", acc.index[-1] - acc.index[0])
print("rows      :", len(acc))

In [ ]:
# The logger stalled somewhere. Find the gaps.
gaps = acc.index.to_series().diff()
print("nominal interval:", gaps.median())
print("\nintervals longer than 3x nominal:")
print(gaps[gaps > 3 * gaps.median()])

> **This is a check to run on every timestamped recording.** A 2-second hole in a 100 Hz
> capture is 200 missing samples. If you feed the array to an FFT as though it were
> contiguous, every frequency you report will be wrong, and nothing will warn you.

In [ ]:
# resample() re-bins onto a regular grid. Downsample with an aggregation:
per_second = acc.resample("1s").agg(["mean", "std", "count"])
print(per_second.head())
print()
print("The 'count' column exposes the gap -- look for a second with far fewer than 100 samples.")
print(per_second[("acc_z", "count")].nsmallest(5))

In [ ]:
# Upsample (or regularise) with asfreq + interpolate.
regular = acc.resample("10ms").mean()          # exactly 100 Hz, gaps become NaN rows
print(f"original {len(acc)} rows -> regularised {len(regular)} rows")
print(f"NaN rows introduced by the gap: {regular['acc_z'].isna().sum()}")

filled = regular.interpolate(limit=5)          # bridge short dropouts only
print(f"after interpolate(limit=5): {filled['acc_z'].isna().sum()} still NaN")

### Common resample rules

| rule | meaning |
|------|---------|
| `"10ms"` | 10 milliseconds → 100 Hz |
| `"1s"`, `"500ms"` | seconds, milliseconds |
| `"1min"`, `"1h"` | minutes, hours |
| `.mean()`, `.max()`, `.std()` | downsampling aggregations |
| `.first()`, `.last()` | pick a representative sample |
| `.asfreq()` | no aggregation — just re-index, NaN where absent |

**Do not use `resample().mean()` as a substitute for `scipy.signal.decimate`.** A box average
is a very poor anti-aliasing filter. Use `resample` to get onto a regular grid; use
`signal.decimate` or `resample_poly` to actually change sampling rate.

---
## 7. Rolling windows

`rolling` gives you a sliding-window computation over a Series — moving averages, envelopes,
running standard deviation for change detection.

In [ ]:
z = filled["acc_z"].interpolate()

roll = pd.DataFrame({
    "raw": z,
    "mean_50": z.rolling(50, center=True).mean(),
    "std_50": z.rolling(50, center=True).std(),
    "max_50": z.rolling(50, center=True).max(),
})

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(roll.index, roll["raw"], "0.8", lw=0.5, label="raw")
ax[0].plot(roll.index, roll["mean_50"], lw=1.4, label="rolling mean (0.5 s)")
ax[0].set_ylabel("acc_z [m/s²]"); ax[0].legend()
ax[1].plot(roll.index, roll["std_50"], color="tab:red", lw=1)
ax[1].set_ylabel("rolling std"); ax[1].set_xlabel("time")
ax[1].set_title("Rolling standard deviation spikes exactly at the vibration burst")
fig.tight_layout()

> **`center=True` matters.** By default a rolling window is *trailing* — the value at index
> `n` summarises samples `n-49` to `n`, so the result is delayed by half a window. This is
> the same causal-vs-zero-phase distinction as `lfilter` vs `filtfilt` in notebook 03.
> `center=True` is the zero-phase version, valid for offline analysis only.

In [ ]:
# Rolling std as a simple event detector.
threshold = roll["std_50"].median() * 3
events = roll["std_50"] > threshold

print(f"threshold: {threshold:.3f}")
print(f"samples flagged: {events.sum()}")

# Group consecutive flagged samples into events.
groups = (events != events.shift()).cumsum()[events]
for gid, idx in pd.Series(groups.index).groupby(groups.to_numpy()):
    if len(idx) > 20:
        print(f"  event: {idx.iloc[0]} -> {idx.iloc[-1]}  ({len(idx)} samples)")

---
## 8. Multi-channel data: `accelerometer.csv`

Three channels, one index. DataFrame operations apply column-wise for free.

In [ ]:
print(acc.describe().round(3))
print()
print("missing values per channel:")
print(acc.isna().sum())

In [ ]:
clean = acc.interpolate(limit=5).dropna()

fig, ax = plt.subplots(3, 1, figsize=(11, 5.5), sharex=True)
for a, col in zip(ax, ["acc_x", "acc_y", "acc_z"]):
    a.plot(clean.index, clean[col], lw=0.4)
    a.set_ylabel(col)
ax[-1].set_xlabel("time")
ax[0].set_title("Three-axis accelerometer -- note the burst around 09:16:01")
fig.tight_layout()

In [ ]:
# Derived columns: magnitude, and gravity-removed vertical acceleration.
clean = clean.assign(
    magnitude=lambda d: np.sqrt(d.acc_x**2 + d.acc_y**2 + d.acc_z**2),
    acc_z_ac=lambda d: d.acc_z - d.acc_z.mean(),
)

print(clean.head())
print()
print("correlation between channels:")
print(clean[["acc_x", "acc_y", "acc_z"]].corr().round(3))

> **`.assign()` with lambdas** builds new columns in a chain without mutating the original,
> and each lambda sees the DataFrame *as it is at that point* — so `acc_z_ac` could refer to
> `magnitude` if it needed to. This is the readable alternative to a wall of
> `df["new"] = ...` statements.

---
## 9. Bridging to SciPy

The handoff. Pull `.to_numpy()`, do the DSP, put the result back as a column.

In [ ]:
fs_acc = 100.0
z = clean["acc_z_ac"].to_numpy()        # <- leave pandas here

sos = signal.butter(6, [15, 35], btype="bandpass", fs=fs_acc, output="sos")
z_band = signal.sosfiltfilt(sos, z)
envelope = np.abs(signal.hilbert(z_band))

clean = clean.assign(z_band=z_band, z_env=envelope)   # <- and come back

fig, ax = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
ax[0].plot(clean.index, clean["acc_z_ac"], lw=0.4)
ax[0].set_ylabel("acc_z (AC)")
ax[1].plot(clean.index, clean["z_band"], lw=0.4, label="15-35 Hz band")
ax[1].plot(clean.index, clean["z_env"], "r", lw=1.2, label="envelope")
ax[1].set_ylabel("filtered"); ax[1].legend()
ax[1].set_xlabel("time")
ax[0].set_title("Band-pass isolates the 24 Hz burst; the envelope locates it in time")
fig.tight_layout()

In [ ]:
# Spectra per channel, assembled into a DataFrame for easy comparison.
spectra = {}
for col in ["acc_x", "acc_y", "acc_z"]:
    f, P = signal.welch(clean[col].to_numpy(), fs_acc, nperseg=512)
    spectra[col] = P
psd = pd.DataFrame(spectra, index=pd.Index(f, name="frequency_hz"))

print(psd.head())
print()
print("dominant frequency per channel:")
print(psd.idxmax().round(2))

ax = psd.plot(logy=True, figsize=(10, 3.2))
ax.set_ylabel("PSD"); ax.set_xlabel("frequency [Hz]")
ax.set_title("Welch PSD per axis (pandas plots all columns at once)")

---
## 10. GroupBy: comparing many experiments

This is where pandas earns its place in a DSP workflow. `data/filter_sweep.csv` holds 750
measurements: 5 filter families × 5 orders × 5 cutoffs × 6 repeated trials, with four metrics
each. Answering "which family gives the best SNR per unit of group delay" from raw NumPy
arrays would be miserable. Here it is three lines.

In [ ]:
sweep = pd.read_csv(f"{DATA}/filter_sweep.csv")
print(sweep.head())
print()
print(f"{len(sweep)} rows")
print(sweep[["family", "order", "cutoff_hz"]].nunique())

In [ ]:
# The core pattern: split -> apply -> combine.
by_family = sweep.groupby("family")["output_snr_db"].agg(["mean", "std", "count"])
print(by_family.round(3).sort_values("mean", ascending=False))

In [ ]:
# Group by several keys, aggregate several columns, each with its own functions.
summary = sweep.groupby(["family", "order"]).agg(
    snr_mean=("output_snr_db", "mean"),
    snr_std=("output_snr_db", "std"),
    delay_mean=("group_delay_ms", "mean"),
    ripple_max=("passband_ripple_db", "max"),
    n=("trial", "count"),
).round(3)

print(summary.head(12))

> **Named aggregation** — `new_name=("column", "function")` — is the readable form. It avoids
> the MultiIndex columns you get from `.agg(["mean", "std"])`, which are correct but painful
> to index into.

In [ ]:
# Transform: attach a group-level statistic back onto every row.
sweep["snr_vs_family_mean"] = (
    sweep["output_snr_db"] - sweep.groupby("family")["output_snr_db"].transform("mean")
)
print(sweep[["family", "order", "output_snr_db", "snr_vs_family_mean"]].head(8).round(3))
print("\n`transform` returns something the same length as the input -- unlike `agg`.")

In [ ]:
# Filter groups by a group-level condition.
low_ripple = sweep.groupby(["family", "order"]).filter(
    lambda g: g["passband_ripple_db"].max() < 0.1
)
print(f"{len(low_ripple)} of {len(sweep)} rows are in (family, order) groups with ripple < 0.1 dB")
print(low_ripple.groupby("family").size())

In [ ]:
# The actual engineering question: best SNR per millisecond of delay.
sweep["efficiency"] = sweep["output_snr_db"] / sweep["group_delay_ms"]
best = (sweep.groupby(["family", "order", "cutoff_hz"])["efficiency"]
             .mean()
             .sort_values(ascending=False)
             .head(10))
print("Top 10 configurations by SNR per ms of group delay:")
print(best.round(3))

---
## 11. Reshaping: `pivot`, `melt`, wide vs. long

**Long (tidy) format**: one row per observation, with columns identifying it. Best for
storage, groupby, and seaborn.

**Wide format**: one row per x-value, one column per series. Best for reading, for heatmaps,
and for matrix maths.

You will convert between them constantly.

In [ ]:
# Long -> wide with pivot_table.
wide = sweep.pivot_table(
    index="order",
    columns="family",
    values="output_snr_db",
    aggfunc="mean",
).round(2)

print("WIDE -- easy to read, one column per family:")
print(wide)

In [ ]:
# Wide -> long with melt.
long = wide.reset_index().melt(
    id_vars="order",
    var_name="family",
    value_name="mean_snr_db",
)
print("LONG -- one row per observation, ready for groupby or seaborn:")
print(long.head(8))

In [ ]:
# A 2-D pivot makes a natural heatmap.
grid = sweep.pivot_table(index="order", columns="cutoff_hz",
                         values="output_snr_db", aggfunc="mean")

fig, ax = plt.subplots(figsize=(7, 3.4))
im = ax.imshow(grid.to_numpy(), aspect="auto", cmap="viridis", origin="lower")
ax.set_xticks(range(len(grid.columns)), grid.columns)
ax.set_yticks(range(len(grid.index)), grid.index)
ax.set_xlabel("cutoff [Hz]"); ax.set_ylabel("order")
ax.set_title("Mean output SNR [dB]"); ax.grid(False)
fig.colorbar(im, ax=ax, label="SNR [dB]")

for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax.text(j, i, f"{grid.iat[i, j]:.0f}", ha="center", va="center",
                color="w", fontsize=8)

---
## 12. Building a results table from scratch

The workflow you will use for every lab report: loop over configurations, measure, collect
dicts, build a DataFrame, sort.

In [ ]:
fs = 1000.0
t = np.arange(2000) / fs
rng = np.random.default_rng(7)
clean_sig = np.sin(2 * np.pi * 10 * t)
noisy_sig = clean_sig + rng.normal(0, 0.5, t.size)

def snr_db(ref, test):
    return 10 * np.log10(np.sum(ref**2) / np.sum((test - ref)**2))

records = []
for family, design in [
    ("butter", lambda n, fc: signal.butter(n, fc, fs=fs, output="sos")),
    ("cheby1", lambda n, fc: signal.cheby1(n, 1.0, fc, fs=fs, output="sos")),
    ("ellip",  lambda n, fc: signal.ellip(n, 1.0, 40.0, fc, fs=fs, output="sos")),
    ("bessel", lambda n, fc: signal.bessel(n, fc, fs=fs, output="sos")),
]:
    for order in [2, 4, 6, 8]:
        for fc in [20.0, 40.0, 80.0]:
            sos = design(order, fc)
            y_causal = signal.sosfilt(sos, noisy_sig)
            y_zero = signal.sosfiltfilt(sos, noisy_sig)
            records.append({
                "family": family,
                "order": order,
                "cutoff_hz": fc,
                "snr_lfilter_db": snr_db(clean_sig, y_causal),
                "snr_filtfilt_db": snr_db(clean_sig, y_zero),
                "n_coeffs": sos.size,
            })

results = pd.DataFrame(records)
results["improvement_db"] = results["snr_filtfilt_db"] - results["snr_lfilter_db"]

print(results.sort_values("snr_filtfilt_db", ascending=False).head(10).round(2))

> **The `improvement_db` column tells the story:** zero-phase filtering buys a large SNR gain
> here purely because the causal filter's delay counts as error against the reference. That
> is a measurement artefact of how we defined SNR, not free performance — and noticing that
> is exactly the kind of thing a results table makes visible.

In [ ]:
print("Best configuration per family (by zero-phase SNR):")
best_per_family = results.loc[results.groupby("family")["snr_filtfilt_db"].idxmax()]
print(best_per_family.round(2).to_string(index=False))

In [ ]:
# Styled output for a report. .style renders with formatting in Jupyter.
(results.groupby(["family", "order"])[["snr_filtfilt_db", "improvement_db"]]
        .mean()
        .round(2)
        .style
        .background_gradient(cmap="Greens", subset=["snr_filtfilt_db"])
        .format("{:.2f}")
        .set_caption("Mean SNR by family and order"))

---
## 13. Exporting

| format | call | when |
|--------|------|------|
| CSV | `.to_csv(path, index=False)` | universal, human-readable, slow and lossy on dtypes |
| Parquet | `.to_parquet(path)` | large data — compressed, typed, fast (needs `pyarrow`) |
| Excel | `.to_excel(path)` | when a colleague insists (needs `openpyxl`) |
| Markdown | `.to_markdown()` | pasting a table into a report or README |
| LaTeX | `.to_latex()` | pasting into a paper |

In [ ]:
import os
os.makedirs("../scratch", exist_ok=True)

summary_table = (results.groupby("family")[["snr_filtfilt_db", "improvement_db"]]
                        .mean().round(2))

summary_table.to_csv("../scratch/filter_results.csv")
print("--- markdown (paste straight into a README) ---")
print(summary_table.to_markdown())
print()
print("--- latex (paste into a report) ---")
print(summary_table.to_latex())

---
## 14. Performance notes

Pandas is fast when you use it vectorised and slow when you loop. Two rules cover most of it.

In [ ]:
big = pd.DataFrame({"x": np.random.default_rng(0).normal(size=200_000)})

print("iterrows (never do this):")
%timeit -n 1 -r 1 sum(row.x ** 2 for _, row in big.head(20_000).iterrows())
print("apply (better, still Python-level):")
%timeit -n 3 -r 3 big["x"].apply(lambda v: v ** 2).sum()
print("vectorised (correct):")
%timeit -n 10 -r 3 (big["x"] ** 2).sum()
print("straight to numpy (fastest):")
%timeit -n 10 -r 3 (big["x"].to_numpy() ** 2).sum()

**Rule 1: never `iterrows`.** It is roughly a thousand times slower than the vectorised form
and there is essentially always a vectorised form.

**Rule 2: drop to NumPy for the actual DSP.** `.to_numpy()` is nearly free (usually a view,
not a copy) and every SciPy function wants an ndarray anyway. Pandas is the container, not
the compute engine.

In [ ]:
# Chained assignment: the classic pandas trap.
d = pd.DataFrame({"a": [1, 2, 3]})

# This may silently fail to modify d (it operates on a temporary copy):
#   d[d["a"] > 1]["a"] = 99
# Do this instead -- one .loc call, one indexing operation:
d.loc[d["a"] > 1, "a"] = 99
print(d)

---
## 15. Exercises

**1. A loader function.** Write `load_bench_capture(path)` that returns a clean DataFrame with
a proper index, bad samples as NaN, and a printed report of how many samples were rejected
and why. Make it raise a clear error if the sample interval is not uniform.

**2. Gap accounting.** For `accelerometer.csv`, produce a table with one row per detected gap:
start time, end time, duration, number of samples missing. Then decide, with a stated
justification, which gaps you would interpolate and which make the surrounding data unusable.

**3. Spectral comparison table.** For each channel of the accelerometer data, compute the
Welch PSD and extract: dominant frequency, total power, power in 0-5 Hz, power in 20-30 Hz,
and spectral centroid. Assemble into one DataFrame and identify which channel best captures
the burst.

**4. Reproduce the sweep.** Regenerate `filter_sweep.csv` yourself: actually design each
filter, actually measure roll-off, ripple, group delay and SNR, and write the results out in
the same tidy format. Compare your measured numbers against the synthetic ones and explain
any structural differences.

**5. Rolling-window features.** For the accelerometer z-axis, compute rolling mean, std,
min-max range, and zero-crossing rate over 1-second windows. Which one separates the burst
from the background most cleanly? Quantify with a ratio of in-burst to out-of-burst values.

**6. Resample carefully.** Take the ECG at 360 Hz and produce a 120 Hz version three ways:
`df.resample()`, `scipy.signal.decimate`, and `scipy.signal.resample_poly`. Compare the
resulting spectra and explain the differences. Which would you defend in a report?

**7. Merge two recordings.** Split `accelerometer.csv` into two DataFrames with different,
offset time bases (e.g. one at 100 Hz and one at 30 Hz). Then use `pd.merge_asof` to align
them onto a common index with a tolerance. This is the real problem when combining sensors
that free-run on separate clocks.

**8. A report generator.** Write a function that takes a raw signal and a sampling rate and
returns a one-row DataFrame of summary statistics (RMS, peak, crest factor, dominant
frequency, bandwidth, SNR estimate). Apply it to every column of every dataset in `data/` and
concatenate into one table.

---

### Quick reference

| Task | Call |
|------|------|
| read CSV | `pd.read_csv(path, parse_dates=[...], na_values=[...])` |
| messy text | `pd.read_csv(path, sep=";", comment="#", header=None, names=[...], skipinitialspace=True)` |
| set index | `df.set_index("time_s")` |
| slice by label | `df.loc[2.0:5.0]` (endpoints included) |
| slice by position | `df.iloc[0:100]` (endpoint excluded) |
| overview | `df.info()`, `df.describe()`, `df.head()` |
| count missing | `df.isna().sum()` |
| fill gaps | `df.interpolate(method="linear", limit=5)` |
| re-bin time | `df.resample("10ms").mean()` |
| sliding window | `s.rolling(50, center=True).std()` |
| new columns | `df.assign(mag=lambda d: ...)` |
| group + aggregate | `df.groupby([...]).agg(name=("col", "mean"))` |
| group stat per row | `df.groupby(k)["c"].transform("mean")` |
| long → wide | `df.pivot_table(index=, columns=, values=, aggfunc=)` |
| wide → long | `df.melt(id_vars=, var_name=, value_name=)` |
| to NumPy | `df["col"].to_numpy()` |
| export | `.to_csv()`, `.to_parquet()`, `.to_markdown()`, `.to_latex()` |